In [ ]:
import pandas as pd

# Première tentative d'appariement au niveau juridique

In [ ]:
MCO_2022 = pd.read_csv("SAE/2022/MCO_2022r.csv", sep=";", encoding='latin1', decimal=',')
HOSPIDIAG = pd.read_csv("Hospidiag/2022/hd2022.csv", sep=";", encoding='latin1', decimal=',')
MCO_2022 = MCO_2022.drop('AN', axis=1).reset_index()
HOSPIDIAG["finess"].nunique()

In [ ]:
MCO_2022_jur = MCO_2022.groupby('FI_EJ').sum(numeric_only=True).reset_index()
MCO_2022_jur["AN"] = 2022
MCO_2022_jur["FI_EJ"].nunique()

In [ ]:
HOSPIDIAG = HOSPIDIAG.rename(columns={'finess': 'FI_EJ'})
MCO_2022_full = MCO_2022_jur.merge(HOSPIDIAG, "left", "FI_EJ")
MCO_2022_full.info()
MCO_2022_full.sample(10)

# Compréhension de la construction des indicateurs au niveau juridique dans Hospidiag issus de SAE

Nous tentons de voir si les indicateurs de personnel dans Hospidiag (finess juridique) recoupent ceux de SAE agrégé par finess juridique. On prend l'exemple de la variable CI_RH1 d'Hospidiag, qui correspond aux ETP médicaux. Précisément, d'après Hospidiag, la variable CI_RH1 de 2022 correspond à : "ETP médicaux, dont Médecins (hors anesthésistes), dont Chirurgiens (hors gynécologues-obstétriciens), dont Anesthésistes, dont Gynécologues-obstétriciens" issus du fichier Q20 de SAE 2022. Par ailleurs, "depuis la SAE 2013, tous les ETP (public et privé) sont désormais estimés à partir des ETP moyens annuels rémunérés."

In [ ]:
hd = pd.read_csv("Hospidiag/2022/hd2022.csv", sep=";", encoding='latin1', decimal=',')
q20 = pd.read_csv("SAE/2022/Q20_2022r.csv", sep=';', encoding='latin1', decimal='.')
q20.head()


In [ ]:
q20_med = q20[~q20["PERSO"].isin(["M3011", "M3020", "M3030", "M3012", "M3050", "M3040", "M3060", "M3070", "M9999"])] 
#sélection de tous les personnels chirurgicaux et médicaux: on exclue les personnels médicaux "autres" de Q20, 
#conformément à la description de CI_RH1 dans hospidiag
q20_med["ETPSAL"] = pd.to_numeric(q20_med["ETPSAL"])
q20_med = q20_med[["FI_EJ", "ETPSAL"]]
q20_med = q20_med.groupby("FI_EJ").sum().reset_index()
q20_med.head()

In [ ]:
hd_etp = hd[["finess","CI_RH1"]]
hd_etp.columns = ["FI_EJ", "CI_RH1"]
hd_etp.shape

In [ ]:
merge = q20_med.merge(hd_etp, how="left", on ="FI_EJ")
merge.sample(10)

On remarque que l'agrégation n'est pas la bonne : ETPSAL et CI_RH1 ne correspondent pas (même quand CI_RH1 est bien défini). On a donc plus d'informations avec ETPSAL du fichier Q20. Cependant, on peut craindre qu'agréger ce fichier par finess juridique ne soit pas suffisant pour apparier hospidiag à Q20, dans la mesure où on ne trouve pas les mêmes résultats.

Prenons une variable "plus simple" pour tenter de voir si dans ce cas l'agrégation de Q20 par finess juridique permet bien de retrouver les mêmes valeurs que dans Hospidiag. CI_RH5 correspond aux ETP médicaux dont gynécologues obstétriciens

In [ ]:
q20_gyn = q20[q20["PERSO"]=="M2050"] 
#sélection de tous les personnels de gynécologie-obstétrique
q20_gyn["ETPSAL"] = pd.to_numeric(q20_gyn["ETPSAL"])
q20_gyn = q20_gyn[["FI_EJ", "ETPSAL"]]
q20_gyn = q20_gyn.groupby("FI_EJ").sum().reset_index()
q20_gyn.head()

hd_etp_gyn = hd[["finess","CI_RH5"]]
hd_etp_gyn.columns = ["FI_EJ", "CI_RH5"]

merge_gyn = q20_gyn.merge(hd_etp_gyn, how="left", on ="FI_EJ")
merge_gyn.sample(10)

Il semblerait que CI_RH1 corresponde finalement à ETPSAL toutes catéogies de personnel confondues (perso M9999). 

In [ ]:
q20_tot = q20[q20["PERSO"]=="M9999"].reset_index()
#sélection du total d'ETP
q20_tot["ETPSAL"] = pd.to_numeric(q20_tot["ETPSAL"])
q20_tot = q20_tot[["FI_EJ", "ETPSAL"]]
q20_tot = q20_tot.groupby("FI_EJ").sum().reset_index()
q20_tot.head()


In [ ]:
merge_tot = q20_tot.merge(hd_etp, how="left", on ="FI_EJ")
print(merge_tot[merge_tot["FI_EJ"]=="630780989"])
merge_tot.head()

On conclut donc que les variables de personnel dans Hospidiag sont bien obtenues par l'agrégation au niveau du finess juridique des variables de la SAE. Cependant, les données sont plus disponibles dans SAE que dans hospidiag. 

In [ ]:
hd.shape[0] == hd["finess"].nunique()

Le finess juridique est bien un identifiant sur lequel on pourrait apparier hospidiag à SAE, une fois seuement les agrégations faites au niveau juridique sur SAE étant donné la forme de la base SAE.